# 예제 03. CNN 모델 구조와 shape 추적
빅데이터프로그래밍 · 8주차

## 목표
- `Conv2d → ReLU → MaxPool2d → Flatten → Linear` 구조를 작성한다
- 각 단계의 shape을 순서대로 출력한다
- Flatten 앞의 숫자를 스스로 계산한다

CNN에서 학생들이 가장 많이 막히는 곳이 **Flatten 뒤 Linear의 입력 크기**입니다.


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)
x = torch.randn(4, 1, 28, 28)      # 데이터 4개
print("입력:", tuple(x.shape))


## 1. 한 단계씩 통과시키며 shape 보기


In [ ]:
conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
relu  = nn.ReLU()
pool  = nn.MaxPool2d(2)
conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)

h = x
for name, layer in [("Conv2d(1→16)", conv1), ("ReLU", relu), ("MaxPool2d(2)", pool),
                    ("Conv2d(16→32)", conv2), ("ReLU", relu), ("MaxPool2d(2)", pool)]:
    h = layer(h)
    print(f"{name:16s} → {tuple(h.shape)}")

print("\nFlatten 하면:", tuple(nn.Flatten()(h).shape))
print("32 × 7 × 7 =", 32 * 7 * 7)


## 2. Flatten 뒤 Linear의 입력 크기 계산
`채널 수 × 높이 × 너비` 입니다. 위 구조에서는 32 × 7 × 7 = 1568.


In [ ]:
import pandas as pd

rows = []
size = 28
ch = 1
rows.append({"단계": "입력", "채널": ch, "크기": f"{size}×{size}", "원소 수": ch*size*size})
for i, out_ch in enumerate([16, 32]):
    ch = out_ch
    rows.append({"단계": f"Conv2d(pad=1) {i+1}", "채널": ch, "크기": f"{size}×{size}", "원소 수": ch*size*size})
    size //= 2
    rows.append({"단계": f"MaxPool2d(2) {i+1}", "채널": ch, "크기": f"{size}×{size}", "원소 수": ch*size*size})
pd.DataFrame(rows)


## 3. 전체 모델 작성


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, n_classes)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))     # 28 → 14
        x = self.pool(torch.relu(self.conv2(x)))     # 14 → 7
        x = self.flatten(x)                          # 32*7*7
        x = torch.relu(self.fc1(x))
        return self.fc2(x)


model = SimpleCNN()
print(model)
print("\n출력:", tuple(model(x).shape))
print("파라미터:", sum(p.numel() for p in model.parameters()))


## 4. Flatten 크기를 틀리면 — 가장 흔한 오류


In [ ]:
bad = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(32 * 14 * 14, 10),     # 7 이어야 하는데 14 로 씀
)
try:
    bad(x)
except RuntimeError as err:
    print("RuntimeError:", err)
    print("\n→ 메시지의 두 숫자 중 앞이 실제 크기, 뒤가 내가 쓴 크기입니다")


## 5. 크기를 자동으로 알아내는 방법
직접 계산하기 번거로울 때 씁니다. 실무에서도 흔히 이렇게 합니다.


In [ ]:
features = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
)

with torch.no_grad():
    n_flat = features(torch.zeros(1, 1, 28, 28)).numel()
print("Flatten 크기:", n_flat)

auto_model = nn.Sequential(features, nn.Flatten(), nn.Linear(n_flat, 10))
print("출력:", tuple(auto_model(x).shape))


In [ ]:
# nn.AdaptiveAvgPool2d 를 쓰면 입력 크기와 무관해집니다
adaptive = nn.Sequential(features, nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, 10))
print("28x28 입력:", tuple(adaptive(torch.randn(2,1,28,28)).shape))
print("40x40 입력:", tuple(adaptive(torch.randn(2,1,40,40)).shape))   # 같은 모델로 처리


## 6. MLP와 파라미터 수 비교
CNN은 필터를 이미지 전체에서 공유하므로 파라미터가 적습니다.


In [ ]:
mlp = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))

rows = []
for name, m in [("MLP (784→128→10)", mlp), ("CNN (16,32 필터)", SimpleCNN())]:
    rows.append({"모델": name, "파라미터 수": sum(p.numel() for p in m.parameters())})
pd.DataFrame(rows)


## 직접 해보기
1. 필터 개수를 (32, 64)로 늘리면 Flatten 크기와 파라미터 수는 어떻게 되나요?
2. Conv를 세 층으로 늘리면 28은 몇까지 줄어드나요? 코드로 확인하세요.


In [ ]:
# 여기에 작성하세요
